## Автоматизация вычислений в конечных полях



### 1. Вероятностный алгоритм поиска образующего элемента циклической группы:



In [1]:
from sage.all import *

# выбираем простое число
prime = 6053
# создаем группу классов вычетов Zp (кольцо Z/pZ)
G = Integers(prime)

# вероятностный алгоритм поиска образующего элемента группы
def find_primitive_element(G):
    # вычисляем порядок группы по умножению
    multiplicative_order = euler_phi(G.order())
    
    # получаем простые делители порядка мультипликативной группы
    divisors = prime_divisors(multiplicative_order)
    while True:
        g = G.random_element()
        if g == 0:
            continue

        for d in divisors:
            if g**(multiplicative_order / d) == 1:
                break
        else:
            return g

g = find_primitive_element(G)
print(g)
assert g.multiplicative_order() == euler_phi(prime) # prime - 1

2278


### 2. Поиск примитивного многочлена поля:



In [3]:
from sage.all import *

x = var('x') # создаем символическую переменную

# создаем основное поле GF(p)
F = GF(2, name='a')
# получаем кольцо многочленов с коэффициентами в GF(p)
R = F[x]

# выбираем необходимую степень неприводимого многочлена
# (степень расширения)
deg = 4

# функция для поиска примитивного элемента поля
def primitive_poly(R, deg):
    while True:
        poly = R.irreducible_element(deg, 'random')
        if poly.is_primitive():
            return poly

print(primitive_poly(R, deg))

x^4 + x + 1


### 3. Деление многочленов в поле:



In [4]:
from sage.all import *

# определяем символическую переменную
x = var('x') 

# определяем конечное поле GF(2)
F = GF(2)
# определяем кольцо многочленов GF(2)[x]
R = F[x]

# определяем многочлен f(x) как элемент кольца многочленов
f = R(x**4 + x + 1)

# задаем список многочленов, на которые хотим поделить
polys = [x+1, x**2 + 1, x**2 + x + 1]
for poly in polys:
    # получаем частное и остаток
    q, r = f.quo_rem(R(poly))
    print(f'f(x) = ({poly})({q}) + {r}')

f(x) = (x + 1)(x^3 + x^2 + x) + 1
f(x) = (x^2 + 1)(x^2 + 1) + x
f(x) = (x^2 + x + 1)(x^2 + x) + 1


### 4. Расчет обратного элемента:

In [5]:
from sage.all import *

x = var('x') # определяем символическую переменную

# определяем конечное поле GF(2^4)
F = GF(2**4, name='x', modulus=x**4 + x + 1)

# итерируемся по всем элементам поля
for el in F:
    # пропускаем 0 и 1
    if el == 0 or el == 1:
        continue
    # находим обратный элемент
    print(f'({el})^(-1) = {el**-1}')

(x)^(-1) = x^3 + 1
(x^2)^(-1) = x^3 + x^2 + 1
(x^3)^(-1) = x^3 + x^2 + x + 1
(x + 1)^(-1) = x^3 + x^2 + x
(x^2 + x)^(-1) = x^2 + x + 1
(x^3 + x^2)^(-1) = x^3 + x
(x^3 + x + 1)^(-1) = x^2 + 1
(x^2 + 1)^(-1) = x^3 + x + 1
(x^3 + x)^(-1) = x^3 + x^2
(x^2 + x + 1)^(-1) = x^2 + x
(x^3 + x^2 + x)^(-1) = x + 1
(x^3 + x^2 + x + 1)^(-1) = x^3
(x^3 + x^2 + 1)^(-1) = x^2
(x^3 + 1)^(-1) = x


### 5. Все элементы как степени корня неприводимого многочлена в поле GF(2^4):

In [6]:
from sage.all import *

x = var('x') # определяем символическую переменную

# определяем конечное поле GF(2^4)
F = GF(2**4, name='a', modulus=x**4 + x + 1)
# получаем корень неприводимого многочлена
a = F.gen()

# перебираем все степени, дающие уникальные элементы
# порядок "a" равен 2**4 - 1 = 15
for i in range(16):
    print(f'a^{i} = {a**i}') 

a^0 = 1
a^1 = a
a^2 = a^2
a^3 = a^3
a^4 = a + 1
a^5 = a^2 + a
a^6 = a^3 + a^2
a^7 = a^3 + a + 1
a^8 = a^2 + 1
a^9 = a^3 + a
a^10 = a^2 + a + 1
a^11 = a^3 + a^2 + a
a^12 = a^3 + a^2 + a + 1
a^13 = a^3 + a^2 + 1
a^14 = a^3 + 1
a^15 = 1


### 6. Таблицы сложения и умножения для поля GF(2^4):

In [7]:
from sage.all import *

x = var('x') # определяем символическую переменную

# определяем конечное поле GF(2^4)
F = GF(2**4, name='a', modulus=x**4 + x + 1)

# функция, которая преобразует элемент в двоичную строку
def element_to_binary(el):
    # получаем коэффициенты многочлена
    coefs = el.polynomial().list()[::-1]
    # добавляем нули до длины 4
    coefs = [0]*(4 - len(coefs)) + coefs
    return ''.join(map(lambda x: str(x), coefs))

# функция, которая выводит таблицу
def print_table(t):
    # получаем ключи для получения исходных элементов
    # в закодированной таблице
    keys = t.column_keys()
    # получаем таблицу в виде двумерного списка
    table = t.table()
    
    # выводим первую строку
    print('    ', end=' ')
    for i in range(len(table)):
        print(element_to_binary(keys[i]), end=' ')
    print()

    for i in range(len(table)):
        print(element_to_binary(keys[i]), end=' ')
        for j in range(len(table)):
            # получаем результат умножения
            el = keys[table[i][j]]
            print(element_to_binary(el), end=' ')
        print()

print("Таблица сложения:")
print_table(F.addition_table())

print("Таблица умножения:")
print_table(F.multiplication_table())

Таблица сложения:
     0000 0001 0010 0011 0100 0101 0110 0111 1000 1001 1010 1011 1100 1101 1110 1111 
0000 0000 0001 0010 0011 0100 0101 0110 0111 1000 1001 1010 1011 1100 1101 1110 1111 
0001 0001 0000 0011 0010 0101 0100 0111 0110 1001 1000 1011 1010 1101 1100 1111 1110 
0010 0010 0011 0000 0001 0110 0111 0100 0101 1010 1011 1000 1001 1110 1111 1100 1101 
0011 0011 0010 0001 0000 0111 0110 0101 0100 1011 1010 1001 1000 1111 1110 1101 1100 
0100 0100 0101 0110 0111 0000 0001 0010 0011 1100 1101 1110 1111 1000 1001 1010 1011 
0101 0101 0100 0111 0110 0001 0000 0011 0010 1101 1100 1111 1110 1001 1000 1011 1010 
0110 0110 0111 0100 0101 0010 0011 0000 0001 1110 1111 1100 1101 1010 1011 1000 1001 
0111 0111 0110 0101 0100 0011 0010 0001 0000 1111 1110 1101 1100 1011 1010 1001 1000 
1000 1000 1001 1010 1011 1100 1101 1110 1111 0000 0001 0010 0011 0100 0101 0110 0111 
1001 1001 1000 1011 1010 1101 1100 1111 1110 0001 0000 0011 0010 0101 0100 0111 0110 
1010 1010 1011 1000 1001 1110 1111 1

### 7. Все элементы как степени корня неприводимого многочлена в поле GF((2^2)^2):

In [8]:
from sage.all import *

x = var('x') # определяем символическую переменную
# определяем поле GF(2^2)
F = GF(2**2, name='b', modulus=x**2 + x + 1)
b = F.gen() # получаем корень неприводимого многочлена

g = var('g') # определяем символическую переменную
# получаем кольцо многочленов с коэффициентами в F
R = F[g]

# определяем неприводимый многочлен x^2 + x + b
# как элемент R
irreducible_poly = R([b, 1, 1])
g = R(g) # определяем g как элемент R

# получаем все ненулевые элементы
for i in range(16):
    print(f'g^{i} = {g**i % irreducible_poly}') 

g^0 = 1
g^1 = g
g^2 = g + b
g^3 = (b + 1)*g + b
g^4 = g + 1
g^5 = b
g^6 = b*g
g^7 = b*g + b + 1
g^8 = g + b + 1
g^9 = b*g + b
g^10 = b + 1
g^11 = (b + 1)*g
g^12 = (b + 1)*g + 1
g^13 = b*g + 1
g^14 = (b + 1)*g + b + 1
g^15 = 1


### 8. Таблицы сложения и умножения для поля GF((2^2)^2):

In [9]:
from sage.all import *

x = var('x') # определяем символическую переменную

# определяем поле GF(2^2)
F = GF(2**2, name='b', modulus=x**2 + x + 1)
b = F.gen() # получаем корень неприводимого многочлена

g = var('g') # определяем символическую переменную
# получаем кольцо многочленов с коэффициентами в F
R = F[g]

# определяем неприводимый многочлен x^2 + x + b
# как элемент R
irr_poly = R([b, 1, 1])
g = R(g) # определяем g как элемент R

# получаем все ненулевые элементы
elements = [g**i % irr_poly for i in range(15)]
elements.insert(0, R(0)) # добавляем ноль

# функция, которая преобразует элемент в двоичную строку
def element_to_binary(el):
    # добавляем нули до длины 2
    coefs = el.coefficients(sparse=False)
    coefs = (coefs + [0] * (2 - len(coefs)))[::-1]
    
    result = []
    for coef in coefs:
        if coef == 0: # ноль не является многочленом
            cfs = [coef]
        else:
            cfs = coef.polynomial().coefficients(sparse=False)
        # добавляем нули до длины 2
        cfs = (cfs + [0] * (2 - len(cfs)))[::-1]
        
        result.extend(cfs)
    return ''.join(map(lambda x: str(x), result))

# сортируем элементы в лексикографическом порядке,
# представляя каждый элемент как бинарную строку
elements.sort(key=lambda x: element_to_binary(x))

# функция, которая выводит таблицу
def print_table(els, op):    
    # выводим первую строку
    print('    ', end=' ')
    for i in range(len(els)):
        print(element_to_binary(els[i]), end=' ')
    print()

    for i in range(len(els)):
        print(element_to_binary(els[i]), end=' ')
        for j in range(len(els)):
            # получаем результат операции op
            el = op(els[i], els[j])
            print(element_to_binary(el), end=' ')
        print()

def elements_sum(el1, el2):
    return (el1 + el2) % irr_poly

def elements_mul(el1, el2):
    return (el1 * el2) % irr_poly


print("Таблица сложения:")
print_table(elements, elements_sum)

print("Таблица умножения:")
print_table(elements, elements_mul)

Таблица сложения:
     0000 0001 0010 0011 0100 0101 0110 0111 1000 1001 1010 1011 1100 1101 1110 1111 
0000 0000 0001 0010 0011 0100 0101 0110 0111 1000 1001 1010 1011 1100 1101 1110 1111 
0001 0001 0000 0011 0010 0101 0100 0111 0110 1001 1000 1011 1010 1101 1100 1111 1110 
0010 0010 0011 0000 0001 0110 0111 0100 0101 1010 1011 1000 1001 1110 1111 1100 1101 
0011 0011 0010 0001 0000 0111 0110 0101 0100 1011 1010 1001 1000 1111 1110 1101 1100 
0100 0100 0101 0110 0111 0000 0001 0010 0011 1100 1101 1110 1111 1000 1001 1010 1011 
0101 0101 0100 0111 0110 0001 0000 0011 0010 1101 1100 1111 1110 1001 1000 1011 1010 
0110 0110 0111 0100 0101 0010 0011 0000 0001 1110 1111 1100 1101 1010 1011 1000 1001 
0111 0111 0110 0101 0100 0011 0010 0001 0000 1111 1110 1101 1100 1011 1010 1001 1000 
1000 1000 1001 1010 1011 1100 1101 1110 1111 0000 0001 0010 0011 0100 0101 0110 0111 
1001 1001 1000 1011 1010 1101 1100 1111 1110 0001 0000 0011 0010 0101 0100 0111 0110 
1010 1010 1011 1000 1001 1110 1111 1